# Cell Counter — Diagnosis Notebook

Run all detection methods on a single chosen image and produce a clean, saveable overview figure.

**Methods covered:**
- **Channel 1 (cells):** Determinant of Hessian (DoH) blob detection
- **Channel 0 (bacteria):** Local Triangle threshold with noise gate

**How to use:**
1. Edit the `⚙️ CONFIGURATION` cell below (file path, time point, parameters)
2. Run all cells (`Run All`)
3. The final cell saves the overview figure to `fig_path`


In [8]:
# ── Imports ───────────────────────────────────────────────────────────────────
from pathlib import Path

import matplotlib
matplotlib.use("Agg")           # non-interactive backend so savefig always works
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

from aicspylibczi import CziFile
from skimage import exposure, transform
from skimage.feature import blob_doh
from skimage.filters import threshold_triangle
from skimage.morphology import remove_small_objects, remove_small_holes

from chipanalysis.utils.file_reader import get_frame, get_pixel_sizes_um, get_timestamps_by_T
from chipanalysis.utils.maye_video_axio import mcherry, gray_cmap
from chipanalysis.chip_alignment import align_chip_to_image, get_roi_from_result, ChipGeometry

%load_ext autoreload
%autoreload 2


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## ⚙️ Configuration

Edit **only this cell** to change the image and parameters.


In [9]:
# ════════════════════════════════════════════════════════════════════════════
#  ⚙️  EDIT EVERYTHING IN THIS CELL
# ════════════════════════════════════════════════════════════════════════════

# ── File & output paths ───────────────────────────────────────────────────────
CZI_PATH = "/Volumes/vangestel/Coco/ZeissData/data_final/AX2+air+bac/CB1-04_S1.czi"

# CZI_PATH  = "/Users/bisot/Documents/PostDoc2/test_data/CB27-01.czi"
FIG_PATH  = "/Users/bisot/Documents/PostDoc2/test_data/fig"   # directory to save figure
FIG_NAME  = "cell_counter_diagnosis.png"                  # saved filename

# ── Image selection ───────────────────────────────────────────────────────────
TIME_POINT   = 0    # time index to analyse

# Channel assignments
CH_CELLS     = 1    # channel containing individual cells (DoH detection)
CH_BACTERIA  = 0    # channel containing bacteria (threshold-based area)

# ── ROI cropping (µm offsets from chip alignment) ─────────────────────────────
ROI_PAD_LEFT_UM   = -1500.0
ROI_PAD_RIGHT_UM  = -3000.0
ROI_PAD_TOP_UM    =  1000.0

# ── Channel 1 — DoH cell detection ───────────────────────────────────────────
CELL_RADIUS_UM    = 5.0    # expected cell radius (µm)
DOH_THRESHOLD     = 0.2    # DoH blob sensitivity (lower = more blobs detected)
STRETCH_MIN_CELLS = 93     # percentile for display contrast stretch (lower bound)
STRETCH_MAX_CELLS = 99.5

# ── Channel 0 — Local Triangle bacteria segmentation ─────────────────────────
TARGET_TILE_SIZE_UM    = 600.0  # local tile size (µm); smaller = finer locality
THRESH_FACTOR          = 0.8    # < 1.0 = more permissive than pure Triangle
SIGNAL_GATE_PERCENTILE = 99.0   # tiles below this global percentile → global floor
MIN_OBJ_UM2            = 10.0   # discard objects smaller than this (µm²)
MAX_HOLE_UM2           = 1.0    # fill holes smaller than this (µm²)
STRETCH_MIN_BAC        = 1
STRETCH_MAX_BAC        = 99.5

# ── Random crop analysis ──────────────────────────────────────────────────────
N_RANDOM_CROPS    = 15          # number of random crops to sample
CROP_SIZE_UM      = 1200.0       # square crop side length (µm)
CROP_RANDOM_SEED  = 42          # for reproducibility

## 1 — Load CZI & align chip


In [11]:
# ── Open CZI ─────────────────────────────────────────────────────────────────
czi        = CziFile(CZI_PATH)
px_um      = get_pixel_sizes_um(czi)["X"]
times      = get_timestamps_by_T(czi, C=0, Z=0)
n_timepoints = len(times)

print(f"File      : {Path(CZI_PATH).name}")
print(f"Pixel size: {px_um:.4f} µm")
print(f"Time pts  : {n_timepoints}  →  analysing t={TIME_POINT}")

# ── Chip alignment (uses brightfield / channel 2 as reference) ───────────────
_ref_img, _ = get_frame(czi, TIME_POINT, 2)
_align      = align_chip_to_image(_ref_img, pixel_size_um=px_um, debug=False, geom=ChipGeometry())

def select_roi(img):
    """Rotate and crop to the chip ROI defined by alignment + padding params."""
    roi, _ = get_roi_from_result(
        _align,
        _align["rotate_fn"](img),
        pad_left_um=ROI_PAD_LEFT_UM,
        pad_right_um=ROI_PAD_RIGHT_UM,
        pad_top_um=ROI_PAD_TOP_UM,
    )
    return roi

print("Chip alignment done ✓")


.//Scaling//Distance[@Id='X']//Value
.//Scaling//Distance[@Id='Y']//Value
File      : CB1-04_S1.czi
Pixel size: 0.9455 µm
Time pts  : 3  →  analysing t=0
Chip alignment done ✓


## 2 — Channel 1: Cell detection (DoH)


In [12]:
# ── Load channel ─────────────────────────────────────────────────────────────
cells_raw, cells_disp = get_frame(
    czi, TIME_POINT, CH_CELLS, gamma=1,
    stretch_min=STRETCH_MIN_CELLS, stretch_max=STRETCH_MAX_CELLS,
)
cells_raw  = select_roi(cells_raw.astype(float))
cells_disp = select_roi(cells_disp)

# ── Normalise & z-score for DoH ───────────────────────────────────────────────
cells_norm = exposure.rescale_intensity(cells_raw, in_range="image", out_range=(0.0, 1.0))
cells_zscore = (cells_norm - cells_norm.mean()) / (cells_norm.std() + 1e-8)

# ── DoH detection ─────────────────────────────────────────────────────────────
sigma_px   = (CELL_RADIUS_UM / px_um) / (2 ** 0.5)
min_sigma  = max(1.0, sigma_px * 0.6)
max_sigma  = max(2.0, sigma_px * 1.4)

cells_blobs = blob_doh(
    cells_zscore,
    min_sigma=min_sigma, max_sigma=max_sigma,
    num_sigma=10, threshold=DOH_THRESHOLD,
)
n_cells = len(cells_blobs)

print(f"Channel {CH_CELLS} (cells) — DoH detection")
print(f"  sigma range : {min_sigma:.2f} – {max_sigma:.2f} px  ({CELL_RADIUS_UM} µm radius)")
print(f"  Cells found : {n_cells}")


Channel 1 (cells) — DoH detection
  sigma range : 2.24 – 5.24 px  (5.0 µm radius)
  Cells found : 1703


## 3 — Channel 0: Bacteria segmentation (local Triangle)


In [13]:
# ── Load channel ─────────────────────────────────────────────────────────────
bac_raw, bac_disp = get_frame(
    czi, TIME_POINT, CH_BACTERIA, gamma=1,
    stretch_min=STRETCH_MIN_BAC, stretch_max=STRETCH_MAX_BAC,
)
bac_raw  = select_roi(bac_raw.astype(float))
bac_disp = select_roi(bac_disp)

bac_norm = exposure.rescale_intensity(bac_raw, out_range=(0.0, 1.0))
H, W = bac_norm.shape

# ── Global floor threshold ────────────────────────────────────────────────────
global_thresh = threshold_triangle(bac_norm) * THRESH_FACTOR
signal_gate   = np.percentile(bac_norm, SIGNAL_GATE_PERCENTILE)

# ── Tile grid from physical size ──────────────────────────────────────────────
img_height_um = H * px_um
img_width_um  = W * px_um
n_tiles_h     = max(1, int(np.ceil(img_height_um / TARGET_TILE_SIZE_UM)))
n_tiles_w     = max(1, int(np.ceil(img_width_um  / TARGET_TILE_SIZE_UM)))
TILE_GRID     = max(n_tiles_h, n_tiles_w)
actual_tile_size_um = img_height_um / TILE_GRID

# ── Per-tile threshold map ────────────────────────────────────────────────────
thresh_map_coarse = np.zeros((TILE_GRID, TILE_GRID), dtype=float)
bac_thresh_map    = np.empty((H, W), dtype=float)
n_gated = 0

for row in range(TILE_GRID):
    for col in range(TILE_GRID):
        r0, r1 = int(row * H / TILE_GRID), int((row + 1) * H / TILE_GRID)
        c0, c1 = int(col * W / TILE_GRID), int((col + 1) * W / TILE_GRID)
        tile = bac_norm[r0:r1, c0:c1]
        local_thresh = threshold_triangle(tile) * THRESH_FACTOR
        if tile.max() < signal_gate:
            t = global_thresh
            n_gated += 1
        else:
            t = local_thresh
        thresh_map_coarse[row, col] = t
        bac_thresh_map[r0:r1, c0:c1] = t

# ── Apply threshold & cleanup ─────────────────────────────────────────────────
bac_binary = bac_norm > bac_thresh_map

min_obj_px  = max(1, int(np.round(MIN_OBJ_UM2  / px_um ** 2)))
max_hole_px = max(1, int(np.round(MAX_HOLE_UM2 / px_um ** 2)))
bac_binary  = remove_small_objects(bac_binary, max_size=min_obj_px)
bac_binary  = remove_small_holes(bac_binary,   max_size=max_hole_px)

# ── Metrics ───────────────────────────────────────────────────────────────────
bac_area_um2 = bac_binary.sum() * px_um ** 2
bac_fraction = bac_binary.mean()

print(f"Channel {CH_BACTERIA} (bacteria) — local Triangle (gated)")
print(f"  Tile grid       : {TILE_GRID}×{TILE_GRID}  (~{actual_tile_size_um:.0f} µm / tile)")
print(f"  Noise gate      : {n_gated}/{TILE_GRID**2} tiles → global floor "
      f"(gate={signal_gate:.4f}, floor={global_thresh:.4f})")
print(f"  Threshold range : {thresh_map_coarse.min():.4f} – {thresh_map_coarse.max():.4f}")
print(f"  Bacteria area   : {bac_area_um2:,.1f} µm²")
print(f"  Coverage        : {bac_fraction * 100:.2f} %")


Channel 0 (bacteria) — local Triangle (gated)
  Tile grid       : 4×4  (~500 µm / tile)
  Noise gate      : 1/16 tiles → global floor (gate=0.1384, floor=0.1016)
  Threshold range : 0.0701 – 0.1216
  Bacteria area   : 206,559.0 µm²
  Coverage        : 6.88 %


## 4 — Overview figure (all methods)


In [14]:
def _make_teal_overlay(img_disp, mask):
    """Return an RGB float image with teal highlight on mask=True pixels."""
    ov = np.stack([img_disp, img_disp, img_disp], axis=-1).astype(float)
    ov[mask, 0] = 0.0
    ov[mask, 1] = ov[mask, 1] * 0.3 + 0.7
    ov[mask, 2] = ov[mask, 2] * 0.3 + 0.7
    return np.clip(ov, 0, 1)


fig = plt.figure(figsize=(16, 10), dpi=150)
fig.patch.set_facecolor("#111111")

# ── Suptitle with key results ─────────────────────────────────────────────────
title = (
    f"{Path(CZI_PATH).name}  |  t={TIME_POINT}"
    f"\nCells (ch{CH_CELLS}, DoH): {n_cells}  ·  "
    f"Bacteria (ch{CH_BACTERIA}, Triangle): coverage={bac_fraction*100:.2f}%  "
    f"area={bac_area_um2:,.0f} µm²"
)
fig.suptitle(title, color="white", fontsize=11, y=0.98)

# ── Layout: 2 rows × 4 columns ────────────────────────────────────────────────
#   Row 0 (cells):     raw | z-score | blob overlay | intensity histogram
#   Row 1 (bacteria):  raw | threshold map | mask overlay | pixel histogram

gs = fig.add_gridspec(2, 4, hspace=0.35, wspace=0.08,
                      left=0.03, right=0.97, top=0.89, bottom=0.04)

axes = [[fig.add_subplot(gs[r, c]) for c in range(4)] for r in range(2)]

_label_kw = dict(color="white", fontsize=8, pad=4)

# ─── Row 0 — Cells (channel 1) ────────────────────────────────────────────────
ax = axes[0][0]
ax.imshow(cells_disp, cmap="gray")
ax.set_title(f"Ch{CH_CELLS} — raw display", **_label_kw)

ax = axes[0][1]
ax.imshow(cells_zscore, cmap="gray")
ax.set_title("Z-score (DoH input)", **_label_kw)

ax = axes[0][2]
ax.imshow(cells_disp, cmap="gray")
for blob in cells_blobs:
    y, x, r = blob
    ax.add_patch(mpatches.Circle((x, y), r, color="yellow",
                                 linewidth=1.0, fill=False))
ax.set_title(f"DoH blobs  (n = {n_cells})", **_label_kw)

ax = axes[0][3]
ax.hist(cells_norm.ravel(), bins=256, color="#f0c040", linewidth=0)
ax.axvline(cells_norm.mean(), color="white", linewidth=1, linestyle="--",
           label=f"mean={cells_norm.mean():.3f}")
ax.set_facecolor("#1a1a1a")
ax.tick_params(colors="white", labelsize=7)
ax.set_title("Intensity histogram", **_label_kw)
ax.legend(fontsize=7, framealpha=0.3, labelcolor="white")
ax.set_xlim(0, 1)

# ─── Row 1 — Bacteria (channel 0) ─────────────────────────────────────────────
ax = axes[1][0]
ax.imshow(bac_disp, cmap="gray")
ax.set_title(f"Ch{CH_BACTERIA} — raw display", **_label_kw)

ax = axes[1][1]
im = ax.imshow(bac_thresh_map, cmap="viridis")
ax.set_title(
    f"Threshold map  ({TILE_GRID}×{TILE_GRID} tiles, ~{actual_tile_size_um:.0f} µm)\n"
    f"gate={signal_gate:.4f}  floor={global_thresh:.4f}  "
    f"({n_gated}/{TILE_GRID**2} gated)",
    **_label_kw,
)
cb = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
cb.ax.tick_params(colors="white", labelsize=6)

ax = axes[1][2]
ax.imshow(_make_teal_overlay(bac_disp, bac_binary))
ax.set_title(
    f"Segmentation mask\ncoverage={bac_fraction*100:.2f}%  area={bac_area_um2:,.0f} µm²",
    **_label_kw,
)

ax = axes[1][3]
ax.hist(bac_norm.ravel(), bins=256, color="#40c0f0", linewidth=0)
ax.axvline(global_thresh, color="red", linewidth=1, linestyle="--",
           label=f"floor={global_thresh:.3f}")
ax.axvline(signal_gate, color="orange", linewidth=1, linestyle=":",
           label=f"gate p{SIGNAL_GATE_PERCENTILE:.0f}={signal_gate:.3f}")
ax.set_facecolor("#1a1a1a")
ax.tick_params(colors="white", labelsize=7)
ax.set_title("Intensity histogram", **_label_kw)
ax.legend(fontsize=7, framealpha=0.3, labelcolor="white")
ax.set_xlim(0, 1)

# ─── Common axis styling ──────────────────────────────────────────────────────
for row in axes:
    for ax in row[:3]:          # image axes
        ax.axis("off")
        ax.set_facecolor("black")
    for spine in row[3].spines.values():  # histogram axes
        spine.set_edgecolor("#444444")

# ── Save ──────────────────────────────────────────────────────────────────────
out_path = Path(FIG_PATH) / FIG_NAME
fig.savefig(out_path, dpi=150, bbox_inches="tight", facecolor=fig.get_facecolor())
print(f"Figure saved → {out_path}")
plt.show()


Figure saved → /Users/bisot/Documents/PostDoc2/test_data/fig/cell_counter_diagnosis.png


/var/folders/lc/xdhphxss45zfz02b14j8dhb00000gp/T/ipykernel_38582/3941015234.py:107: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 5 — Random crop analysis (repeatability test)

In [37]:
np.random.seed(CROP_RANDOM_SEED)

# ── Load full frames (no ROI crop) ────────────────────────────────────────────
bac_full_raw,   _ = get_frame(czi, TIME_POINT, CH_BACTERIA,  gamma=1,
                               stretch_min=STRETCH_MIN_BAC,   stretch_max=STRETCH_MAX_BAC)
cells_full_raw, _ = get_frame(czi, TIME_POINT, CH_CELLS,     gamma=1,
                               stretch_min=STRETCH_MIN_CELLS, stretch_max=STRETCH_MAX_CELLS)
bac_full_raw   = bac_full_raw.astype(float)
cells_full_raw = cells_full_raw.astype(float)

H_full, W_full = bac_full_raw.shape
crop_size_px   = int(np.round(CROP_SIZE_UM / px_um))

print(f"Full frame : {H_full}×{W_full} px  ({H_full*px_um:.0f}×{W_full*px_um:.0f} µm)")
print(f"Crop size  : {crop_size_px}×{crop_size_px} px  ({CROP_SIZE_UM:.0f} µm)")

max_r = H_full - crop_size_px
max_c = W_full - crop_size_px
if max_r <= 0 or max_c <= 0:
    raise ValueError(f"Crop size ({crop_size_px} px) exceeds image dimensions ({H_full}×{W_full})")

crop_rows = np.random.randint(0, max_r + 1, N_RANDOM_CROPS)
crop_cols = np.random.randint(0, max_c + 1, N_RANDOM_CROPS)

czi_stem = Path(CZI_PATH).stem
out_dir  = Path(FIG_PATH)
out_dir.mkdir(parents=True, exist_ok=True)


# ── Helper: run both analyses on a single crop ────────────────────────────────
def analyze_crop(bac_crop_raw, cells_crop_raw):
    """
    Mirrors Section 3 (cell 10) exactly: each crop is self-contained —
    normalised to its own [0,1] range, global_thresh and signal_gate
    derived from that normalised crop, tiles gated against it.
    """
    # ── Bacteria: local Triangle (identical to Section 3) ─────────────────────
    bac_norm = exposure.rescale_intensity(bac_crop_raw, out_range=(0.0, 1.0))
    H_c, W_c = bac_norm.shape

    # Display image (per-crop contrast stretch, for visualisation only)
    bac_disp_crop = exposure.rescale_intensity(bac_crop_raw, in_range=(
        np.percentile(bac_crop_raw, STRETCH_MIN_BAC),
        np.percentile(bac_crop_raw, STRETCH_MAX_BAC),
    ), out_range=(0.0, 1.0)).clip(0, 1)

    # Global floor & gate — same as Section 3, just on the crop
    c_global_thresh = threshold_triangle(bac_norm) * THRESH_FACTOR
    c_signal_gate   = np.percentile(bac_norm, SIGNAL_GATE_PERCENTILE)

    # Tile grid from physical size
    img_h_um  = H_c * px_um
    img_w_um  = W_c * px_um
    n_tiles_h = max(1, int(np.ceil(img_h_um / TARGET_TILE_SIZE_UM)))
    n_tiles_w = max(1, int(np.ceil(img_w_um / TARGET_TILE_SIZE_UM)))
    tile_grid = max(n_tiles_h, n_tiles_w)

    thresh_coarse = np.zeros((tile_grid, tile_grid), dtype=float)
    thresh_map_c  = np.empty((H_c, W_c), dtype=float)
    n_gated_c = 0
    for row in range(tile_grid):
        for col in range(tile_grid):
            r0, r1 = int(row * H_c / tile_grid), int((row + 1) * H_c / tile_grid)
            c0, c1 = int(col * W_c / tile_grid), int((col + 1) * W_c / tile_grid)
            tile = bac_norm[r0:r1, c0:c1]
            local_thresh = threshold_triangle(tile) * THRESH_FACTOR
            if tile.max() < c_signal_gate:
                t = c_global_thresh
                n_gated_c += 1
            else:
                t = local_thresh
            thresh_coarse[row, col] = t
            thresh_map_c[r0:r1, c0:c1] = t
    binary = bac_norm > thresh_map_c
    min_obj_px_c  = max(1, int(np.round(MIN_OBJ_UM2  / px_um ** 2)))
    max_hole_px_c = max(1, int(np.round(MAX_HOLE_UM2 / px_um ** 2)))
    binary = remove_small_objects(binary, max_size=min_obj_px_c)
    binary = remove_small_holes(binary,   max_size=max_hole_px_c)

    coverage_pct = binary.mean() * 100.0
    area_um2     = binary.sum()  * px_um ** 2

    # ── Cells: DoH ───────────────────────────────────────────────────────────
    cells_norm = exposure.rescale_intensity(cells_crop_raw, in_range="image", out_range=(0.0, 1.0))
    cells_disp_crop = exposure.rescale_intensity(cells_crop_raw, in_range=(
        np.percentile(cells_crop_raw, STRETCH_MIN_CELLS),
        np.percentile(cells_crop_raw, STRETCH_MAX_CELLS),
    ), out_range=(0.0, 1.0)).clip(0, 1)
    cells_zscore = (cells_norm - cells_norm.mean()) / (cells_norm.std() + 1e-8)

    sigma_px_c = (CELL_RADIUS_UM / px_um) / (2 ** 0.5)
    try:
        blobs = blob_doh(cells_zscore,
                         min_sigma=max(1.0, sigma_px_c * 0.6),
                         max_sigma=max(2.0, sigma_px_c * 1.4),
                         num_sigma=10, threshold=DOH_THRESHOLD)
    except Exception:
        blobs = np.empty((0, 3))

    return dict(
        bac_disp=bac_disp_crop, bac_norm=bac_norm,
        thresh_map=thresh_map_c, binary=binary,
        global_thresh=c_global_thresh, signal_gate=c_signal_gate,
        n_gated=n_gated_c, tile_grid=tile_grid,
        thresh_range=(thresh_coarse.min(), thresh_coarse.max()),
        cells_disp=cells_disp_crop, cells_zscore=cells_zscore,
        blobs=blobs,
        n_cells=len(blobs), coverage_pct=coverage_pct, area_um2=area_um2,
    )


# ── Run analysis & save one figure per crop ───────────────────────────────────
print(f"\nProcessing {N_RANDOM_CROPS} crops…")

for i, (r, c) in enumerate(zip(crop_rows, crop_cols)):
    bac_crop   = bac_full_raw  [r:r+crop_size_px, c:c+crop_size_px]
    cells_crop = cells_full_raw[r:r+crop_size_px, c:c+crop_size_px]

    res = analyze_crop(bac_crop, cells_crop)

    # ── Per-crop figure: 2 rows × 3 columns ──────────────────────────────────
    #   Row 0 (cells):    raw display | z-score | DoH blob overlay
    #   Row 1 (bacteria): raw display | threshold map | mask overlay
    fig_c, axs = plt.subplots(2, 3, figsize=(9, 6), dpi=120)
    fig_c.patch.set_facecolor("#111111")
    _kw = dict(color="white", fontsize=8, pad=3)

    axs[0, 0].imshow(res["cells_disp"], cmap="gray")
    axs[0, 0].set_title(f"Ch{CH_CELLS} raw", **_kw)

    axs[0, 1].imshow(res["cells_zscore"], cmap="gray")
    axs[0, 1].set_title("Z-score (DoH input)", **_kw)

    axs[0, 2].imshow(res["cells_disp"], cmap="gray")
    for blob in res["blobs"]:
        by, bx, br = blob
        axs[0, 2].add_patch(mpatches.Circle((bx, by), br, color="yellow",
                                             linewidth=0.8, fill=False))
    axs[0, 2].set_title(f"DoH blobs  n={res['n_cells']}", **_kw)

    axs[1, 0].imshow(res["bac_disp"], cmap="gray")
    axs[1, 0].set_title(f"Ch{CH_BACTERIA} raw", **_kw)

    im_thresh = axs[1, 1].imshow(res["thresh_map"], cmap="viridis")
    axs[1, 1].set_title(
        f"Threshold map  ({res['tile_grid']}×{res['tile_grid']} tiles)\n"
        f"gate={res['signal_gate']:.4f}  floor={res['global_thresh']:.4f}  "
        f"({res['n_gated']}/{res['tile_grid']**2} gated)", **_kw)
    cb = fig_c.colorbar(im_thresh, ax=axs[1, 1], fraction=0.046, pad=0.04)
    cb.ax.tick_params(colors="white", labelsize=6)

    axs[1, 2].imshow(_make_teal_overlay(res["bac_disp"], res["binary"]))
    axs[1, 2].set_title(
        f"Mask  cov={res['coverage_pct']:.2f}%\narea={res['area_um2']:,.0f} µm²", **_kw)

    for ax in axs.flat:
        ax.axis("off")
        ax.set_facecolor("black")

    fig_c.suptitle(
        f"{czi_stem}  t={TIME_POINT}  crop {i:03d}  "
        f"[r={r}, c={c}] ({r*px_um:.0f}, {c*px_um:.0f} µm)",
        color="white", fontsize=9, y=1.01,
    )
    fig_c.tight_layout()

    crop_fig_path = out_dir / f"{czi_stem}_t{TIME_POINT}_crop{i:03d}.png"
    fig_c.savefig(crop_fig_path, dpi=120, bbox_inches="tight",
                  facecolor=fig_c.get_facecolor())
    plt.close(fig_c)

    if (i + 1) % 5 == 0 or i == N_RANDOM_CROPS - 1:
        print(f"  {i+1}/{N_RANDOM_CROPS}  saved → {crop_fig_path.name}")

print("Done.")


Full frame : 4512×8576 px  (3219×6118 µm)
Crop size  : 1682×1682 px  (1200 µm)

Processing 15 crops…
  5/15  saved → CB27-01_t0_crop004.png
  10/15  saved → CB27-01_t0_crop009.png
  15/15  saved → CB27-01_t0_crop014.png
Done.
